In [1]:
import glob
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import numpy as np
from regions import Regions
from astropy.nddata import Cutout2D
import astropy.units as u
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from astropy.table import Table, vstack

f140m_nrca_catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f140m_nrca_indivexp_merged_dao_refined.fits')
f140m_nrcb_catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f140m_nrcb_indivexp_merged_dao_refined.fits')
f480m_nrca_catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f480m_nrca_indivexp_merged_dao_refined.fits')
f480m_nrcb_catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f480m_nrcb_indivexp_merged_dao_refined.fits')


f140m_nrca_skycoord = f140m_nrca_catalog['skycoord']
f140m_nrcb_skycoord = f140m_nrcb_catalog['skycoord']
f480m_nrca_skycoord = f480m_nrca_catalog['skycoord']
f480m_nrcb_skycoord = f480m_nrcb_catalog['skycoord']

idx, d2d, d3d = f140m_nrca_skycoord.match_to_catalog_sky(f140m_nrcb_skycoord)

fwhm_tbl = Table.read('/orange/adamginsburg/jwst/w51/reduction/fwhm_table.ecsv')
row = fwhm_tbl[fwhm_tbl['Filter'] == 'F140M']
fwhm = float(row['PSF FWHM (arcsec)'][0]) *u.arcsec

match_f140m = d2d < fwhm*2

new_nrcb = np.setdiff1d(np.arange(len(f140m_nrcb_skycoord)), idx[match_f140m])

print(len(f140m_nrca_catalog), len(f140m_nrcb_catalog), len(new_nrcb))
f140m_catalog = vstack([f140m_nrca_catalog, f140m_nrcb_catalog[new_nrcb]])
print(len(f140m_catalog))

idx, d2d, d3d = f480m_nrca_skycoord.match_to_catalog_sky(f480m_nrcb_skycoord)
row = fwhm_tbl[fwhm_tbl['Filter'] == 'F480M']
fwhm = float(row['PSF FWHM (arcsec)'][0]) *u.arcsec

match_f480m = d2d < fwhm*2
new_nrcb = np.setdiff1d(np.arange(len(f480m_nrcb_skycoord)), idx[match_f480m])

print(len(f480m_nrca_catalog), len(f480m_nrcb_catalog), len(new_nrcb))
f480m_catalog = vstack([f480m_nrca_catalog, f480m_nrcb_catalog[new_nrcb]])
print(len(f480m_catalog))




idx, d2d, d3d = f140m_catalog['skycoord'].match_to_catalog_sky(f480m_catalog['skycoord'])
reverse_idx, reverse_d2d, reverse_d3d = f480m_catalog['skycoord'].match_to_catalog_sky(f140m_catalog['skycoord'])

row = fwhm_tbl[fwhm_tbl['Filter'] == 'F140M']
fwhm = float(row['PSF FWHM (arcsec)'][0]) *u.arcsec
match_f140m_f480m = d2d < fwhm*2
new_f480m = np.setdiff1d(np.arange(len(f480m_catalog)), idx[match_f140m_f480m])
print(len(f140m_catalog), len(f480m_catalog), len(new_f480m))

final_catalog = Table()
final_catalog['skycoord'] = SkyCoord(np.concatenate([f140m_catalog['skycoord'].ra, f480m_catalog['skycoord'][new_f480m].ra]),
                                     np.concatenate([f140m_catalog['skycoord'].dec, f480m_catalog['skycoord'][new_f480m].dec]), frame='icrs', unit='deg')
print(len(final_catalog))
print(idx[match_f140m_f480m])
print(len(idx))
print(len(idx[match_f140m_f480m]))
for col in ['flux_fit', 'flux_err', 'nmatch_good', 'qfit', 'cfit', 'sharpness', 'roundness1', 'roundness2']:
    final_catalog[col+'_f140m'] = np.nan
    final_catalog[col+'_f140m'][:len(f140m_catalog)] = f140m_catalog[col]
    final_catalog[col+'_f480m'] = np.nan
    final_catalog[col+'_f480m'][np.arange(len(f140m_catalog))[match_f140m_f480m]] = f480m_catalog[col][idx[match_f140m_f480m]]
    final_catalog[col+'_f480m'][len(f140m_catalog):] = f480m_catalog[col][new_f480m]
final_catalog.pprint(max_width=200)



8072 7907 7612
15684
8523 6030 5979
14502
15684 14502 7849
23533
[ 4917  5774  4931 ... 13695 13696 13855]
15684
6689
               skycoord                 flux_fit_f140m     flux_fit_f480m     flux_err_f140m   ...    roundness1_f140m      roundness1_f480m      roundness2_f140m      roundness2_f480m  
               deg,deg                                                                         ...                                                                                        
------------------------------------- ------------------ ------------------ ------------------ ... --------------------- --------------------- --------------------- ---------------------
290.93077021040165,14.539893632625967  36.77225080611347                nan 0.6047068104128598 ...   0.09015686886641194                   nan  0.037530401965367076                   nan
290.93209441197206,14.536869819024586 121.68678431662349 21.050013702869716 0.8366005436013549 ...  -0.06470027566758024   -0.38604702

In [2]:
# merge across filters
catalogs_nircam = {"f140m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f140m_nrca_indivexp_merged_dao_basic.fits',
                   "f162m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f162m_nrca_indivexp_merged_dao_basic.fits',
                   "f182m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f182m_nrca_indivexp_merged_dao_basic.fits',
                   "f187n_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f187n_nrca_indivexp_merged_dao_basic.fits',
                   "f210m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f210m_nrca_indivexp_merged_dao_basic.fits',
                   "f335m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f335m_nrca_indivexp_merged_dao_basic.fits',
                   "f360m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f360m_nrca_indivexp_merged_dao_basic.fits',
                   "f405n_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f405n_nrca_indivexp_merged_dao_basic.fits',
                   "f410m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f410m_nrca_indivexp_merged_dao_basic.fits',
                   "f480m_nrca": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f480m_nrca_indivexp_merged_dao_basic.fits',
                   "f140m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f140m_nrcb_indivexp_merged_dao_basic.fits',
                   "f162m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f162m_nrcb_indivexp_merged_dao_basic.fits',
                   "f182m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f182m_nrcb_indivexp_merged_dao_basic.fits',
                   "f187n_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f187n_nrcb_indivexp_merged_dao_basic.fits',
                   "f210m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f210m_nrcb_indivexp_merged_dao_basic.fits',
                   "f335m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f335m_nrcb_indivexp_merged_dao_basic.fits',
                   "f360m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f360m_nrcb_indivexp_merged_dao_basic.fits',
                   "f405n_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f405n_nrcb_indivexp_merged_dao_basic.fits',
                   "f410m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f410m_nrcb_indivexp_merged_dao_basic.fits',
                   "f480m_nrcb": '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/f480m_nrcb_indivexp_merged_dao_basic.fits'}

for band in ['f162m', 'f182m', 'f187n', 'f210m', 'f335m', 'f360m', 'f405n', 'f410m' ]:
    nrca_catalog = Table.read(catalogs_nircam[band+'_nrca'])
    nrcb_catalog = Table.read(catalogs_nircam[band+'_nrcb'])
    nrca_skycoord = nrca_catalog['skycoord']
    nrcb_skycoord = nrcb_catalog['skycoord']
    idx, d2d, d3d = nrca_skycoord.match_to_catalog_sky(nrcb_skycoord)
    row = fwhm_tbl[fwhm_tbl['Filter'] == band.upper()]
    fwhm = float(row['PSF FWHM (arcsec)'][0]) * u.arcsec
    match = d2d < fwhm*2
    new_nrcb = np.setdiff1d(np.arange(len(nrcb_skycoord)), idx[match])
    catalog = vstack([nrca_catalog, nrcb_catalog[new_nrcb]])
    print(band, len(nrca_catalog), len(nrcb_catalog), len(new_nrcb), len(catalog))

    idx, d2d, d3d = final_catalog['skycoord'].match_to_catalog_sky(catalog['skycoord'])
    match = d2d < fwhm*2
    idx_reverse, d2d_reverse, d3d_reverse = catalog['skycoord'].match_to_catalog_sky(final_catalog['skycoord'])
    for col in ['flux_fit', 'flux_err', 'nmatch_good', 'qfit', 'cfit', 'sharpness', 'roundness1', 'roundness2']:
        final_catalog[col+'_'+band] = np.nan
        final_catalog[col+'_'+band][match] = catalog[col][idx[match]]
    
print(len(final_catalog))
final_catalog['nmatch_bands'] = np.sum(~np.isnan(final_catalog['flux_fit_'+band]) for band in ['f140m', 'f162m', 'f182m', 'f187n', 'f210m', 'f335m', 'f360m', 'f405n', 'f410m', 'f480m'])

f162m 119517 116565 113111 232628


f182m 118161 132018 127864 246025


f187n 85188 105890 102453 187641


f210m 119018 121640 117526 236544


f335m 130892 131188 130467 261359


f360m 116203 122891 122191 238394


f405n 65036 80799 80148 145184


f410m 87064 94194 93524 180588
23533


/scratch/local/14626337/ipykernel_1577515/2607033576.py:44: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  final_catalog['nmatch_bands'] = np.sum(~np.isnan(final_catalog['flux_fit_'+band]) for band in ['f140m', 'f162m', 'f182m', 'f187n', 'f210m', 'f335m', 'f360m', 'f405n', 'f410m', 'f480m'])


In [3]:
final_catalog.write('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/final_nircam_indivexp_merged_dao_refined.fits', overwrite=True)